### This code is to implement SCD Type 2

In [0]:
# Assign libraries
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
# We are dealing with Patient Data. This is a existing patient data
existing_data = [
    (1, "John", "Doe", "M", 30, "2025-01-01", None, "Y"),
    (2, "Mary", "James", "F", 25, "2025-01-01", None, "Y")
]

existing_schema = StructType([
    StructField("patient_id", IntegerType(), True),
    StructField("first_name", StringType(), True),
    StructField("last_name", StringType(), True),
    StructField("gender", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("start_date", StringType(), True),
    StructField("end_date", StringType(), True),
    StructField("is_current", StringType(), True)
])

dim_df = spark.createDataFrame(existing_data, existing_schema)

dim_df.show()

In [0]:
#Incoming source data

new_data = [
    (1, "John", "Doe", "M", 31),
    (2, "Mary", "James", "F", 25),
    (3, "Sam", "Wilson", "M", 40)
]

new_schema = StructType([
    StructField("patient_id", IntegerType(), True),
    StructField("first_name", StringType(), True),
    StructField("last_name", StringType(), True),
    StructField("gender", StringType(), True),
    StructField("age", IntegerType(), True)])

source_df = spark.createDataFrame(new_data, new_schema)

source_df.show()

**Identify Changed Records**

In [0]:
#Join current active records with source data.

joined_df = source_df.alias("src").join(
    dim_df.filter(col("is_current") == "Y").alias("dim"),
    on="patient_id",
    how="left"
)
display(joined_df)

In [0]:
# Find changed records

changed_df = joined_df.filter(
    (col("dim.age") != col("src.age")) |
    (col("dim.first_name") != col("src.first_name")) |
    (col("dim.last_name") != col("src.last_name")) |
    (col("dim.gender") != col("src.gender"))
)

display(changed_df)

**Expire Old Records**

In [0]:
# Old versions should become inactive.

expired_df = changed_df.select(
    col("dim.patient_id"),
    col("dim.first_name"),
    col("dim.last_name"),
    col("dim.gender"),
    col("dim.age"),
    col("dim.start_date"),
    current_date().alias("end_date"),
    lit("N").alias("is_current")
)

#display(expired_df)

**Insert New Versions**

In [0]:
# Create new active versions of existing record

new_version_df = changed_df.select(
    col("src.patient_id"),
    col("src.first_name"),
    col("src.last_name"),
    col("src.gender"),
    col("src.age"),
    current_date().alias("start_date"),
    lit(None).cast("date").alias("end_date"),
    lit("Y").alias("is_current")
)

display(new_version_df)

**Identify Completely New Records**

In [0]:
# New Records marked as Active

new_records_df = joined_df.filter(col("dim.patient_id").isNull()) \
.select(
    col("src.patient_id"),
    col("src.first_name"),
    col("src.last_name"),
    col("src.gender"),
    col("src.age"),
    current_date().alias("start_date"),
    lit(None).cast("date").alias("end_date"),
    lit("Y").alias("is_current")
)

display(new_records_df)

**Keep Unchanged Existing Records**

In [0]:
# Unchanged records kept them as is
unchanged_df = dim_df.join(
    changed_df.select("src.patient_id"),
    dim_df.patient_id == col("src.patient_id"),
    "left_anti"
)

display(unchanged_df)

**Final SCD Table**

In [0]:
# Final dataframe after SCD Type 2 implementation
final_df = (
    unchanged_df
    .unionByName(expired_df)
    .unionByName(new_version_df)
    .unionByName(new_records_df)
)

final_df.show(truncate=False)